In [ ]:
# !wget https://www.ncei.noaa.gov/data/global-hourly/archive/csv/2024.tar.gz

In [ ]:
# !mkdir 2024_Dataset
# !tar xf 2024.tar.gz -C 2024_Dataset

In [ ]:
# !rm 2024.tar.gz

## Load Raw Data

In [1]:
import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession \
       .builder \
       .appName("DSA5208_Proj2") \
       .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/29 13:53:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/10/29 13:53:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
print("Spark master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
spark.sparkContext.getConf().getAll()

Spark master: local[*]
Default parallelism: 12


[('spark.driver.extraJavaOptions',
  '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false'),
 ('spark.app.id', 'local-1761731763263'),
 ('spark.executor.id', 'driver'),
 ('spark.driver.port', '409

In [ ]:
df = spark.read.csv("2024_Dataset/", header=True, inferSchema=True)

### Basic Selection

We select the necessary columns, drop columns that are not meaningful and drop rows with NA values. 

In [3]:
df = df.select("STATION",
               "DATE",
               "SOURCE",
               "LATITUDE",
               "LONGITUDE",
               "ELEVATION",
               "NAME",
               "REPORT_TYPE",
               "CALL_SIGN",
               "QUALITY_CONTROL",
               "WND",
               "CIG",
               "VIS",
               "TMP",
               "DEW",
               "SLP")

In [5]:
df.show(20)

+-----------+-------------------+------+--------+---------+---------+--------------------+-----------+---------+---------------+--------------+-----------+------------+-------+-------+-------+
|    STATION|               DATE|SOURCE|LATITUDE|LONGITUDE|ELEVATION|                NAME|REPORT_TYPE|CALL_SIGN|QUALITY_CONTROL|           WND|        CIG|         VIS|    TMP|    DEW|    SLP|
+-----------+-------------------+------+--------+---------+---------+--------------------+-----------+---------+---------------+--------------+-----------+------------+-------+-------+-------+
|99999963894|2024-01-01T00:00:00|     I| 34.7728| -87.6399|    161.5|MUSCLE SHOALS 2 N...|      CRN05|    99999|           V020|211,1,H,0018,1|99999,9,9,N|999999,9,9,9|+0105,1|+9999,9|99999,9|
|99999963894|2024-01-01T00:05:00|     I| 34.7728| -87.6399|    161.5|MUSCLE SHOALS 2 N...|      CRN05|    99999|           V020|214,1,H,0017,1|99999,9,9,N|999999,9,9,9|+0100,1|+9999,9|99999,9|
|99999963894|2024-01-01T00:10:00|  

In [6]:
df.groupBy("CALL_SIGN").count().orderBy(F.desc("count")).show(10)

+---------+--------+
|CALL_SIGN|   count|
+---------+--------+
|    99999|97731802|
|    KAFP |   25588|
|    KIXA |   25551|
|    KEYF |   25547|
|    KEDE |   25544|
|    KCPC |   25448|
|    KEXX |   25448|
|    KFME |   25426|
|    KCGE |   25425|
|    KONX |   25412|
+---------+--------+
only showing top 10 rows



We remove Source, Name, call sign, report type or station number as these are identifiers/specification of sensors or reports, as these are not desired predictive features for our model. Quality Control, which is a specification of Quality control system is also removed.

In [4]:
df = df.select("DATE",
               "LATITUDE",
               "LONGITUDE",
               "ELEVATION",
               "WND",
               "CIG",
               "VIS",
               "TMP",
               "DEW",
               "SLP")

In [7]:
df.printSchema()

root
 |-- DATE: string (nullable = true)
 |-- LATITUDE: string (nullable = true)
 |-- LONGITUDE: string (nullable = true)
 |-- ELEVATION: string (nullable = true)
 |-- WND: string (nullable = true)
 |-- CIG: string (nullable = true)
 |-- VIS: string (nullable = true)
 |-- TMP: string (nullable = true)
 |-- DEW: string (nullable = true)
 |-- SLP: string (nullable = true)



In [9]:
df_cleaned = df.na.drop()
print(f"Number of rows before NA drops: {df.count()}")
print(f"Number of rows after NA drops: {df_cleaned.count()}")

Number of rows before NA drops: 130222106


Number of rows after NA drops: 130222106


Note that there are no NULL values. However, we have to manually account for values (such as 99999.0) that denote missing/null values as per the documentation. 

## Data Parsing & Casting

### DateTime

In [5]:
# Generate the fields for month, day. hour, minute. We drop the year since all records are from 2024

DATE_FORMAT = "yyyy-MM-dd'T'HH:mm:ss"

df = df.withColumn("formatted_timestamp", F.to_timestamp(F.col("DATE"), DATE_FORMAT)) \
              .withColumn("MONTH", F.month("formatted_timestamp")) \
              .withColumn("DAY", F.dayofmonth("formatted_timestamp")) \
              .withColumn("HOUR", F.hour("formatted_timestamp")) \
              .withColumn("MINUTE", F.minute("formatted_timestamp")) \
              .drop("formatted_timestamp")\
              .drop("DATE")

df.show(20)

+--------+---------+---------+--------------+-----------+------------+-------+-------+-------+-----+---+----+------+
|LATITUDE|LONGITUDE|ELEVATION|           WND|        CIG|         VIS|    TMP|    DEW|    SLP|MONTH|DAY|HOUR|MINUTE|
+--------+---------+---------+--------------+-----------+------------+-------+-------+-------+-----+---+----+------+
| 34.7728| -87.6399|    161.5|211,1,H,0018,1|99999,9,9,N|999999,9,9,9|+0105,1|+9999,9|99999,9|    1|  1|   0|     0|
| 34.7728| -87.6399|    161.5|214,1,H,0017,1|99999,9,9,N|999999,9,9,9|+0100,1|+9999,9|99999,9|    1|  1|   0|     5|
| 34.7728| -87.6399|    161.5|212,1,H,0017,1|99999,9,9,N|999999,9,9,9|+0088,1|+9999,9|99999,9|    1|  1|   0|    10|
| 34.7728| -87.6399|    161.5|209,1,H,0018,1|99999,9,9,N|999999,9,9,9|+0077,1|+9999,9|99999,9|    1|  1|   0|    15|
| 34.7728| -87.6399|    161.5|208,1,H,0016,1|99999,9,9,N|999999,9,9,9|+0076,1|+9999,9|99999,9|    1|  1|   0|    20|
| 34.7728| -87.6399|    161.5|209,1,H,0016,1|99999,9,9,N|999999,

### Long, Lat, Elevation

In [6]:
# replace column with casted types

df = df.withColumn("LATITUDE", F.col("LATITUDE").cast(T.FloatType()))\
    .withColumn("LONGITUDE", F.col("LONGITUDE").cast(T.FloatType()))\
    .withColumn("ELEVATION", F.col("ELEVATION").cast(T.FloatType()))


df.printSchema()

root
 |-- LATITUDE: float (nullable = true)
 |-- LONGITUDE: float (nullable = true)
 |-- ELEVATION: float (nullable = true)
 |-- WND: string (nullable = true)
 |-- CIG: string (nullable = true)
 |-- VIS: string (nullable = true)
 |-- TMP: string (nullable = true)
 |-- DEW: string (nullable = true)
 |-- SLP: string (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- HOUR: integer (nullable = true)
 |-- MINUTE: integer (nullable = true)



### Wind

In [7]:
#format WND colum 
# Item 0 is direction, item 2 is a CODE, item 3 is speed]

df_wind_split = df.withColumn("WND", F.split(F.col("WND"), ",")).\
    withColumn("WND_DIR", F.col("WND").getItem(0).cast(T.FloatType()))\
    .withColumn("WND_DIR_QUALITY", F.col("WND").getItem(1).cast(T.IntegerType()))\
    .withColumn("WND_DIR_CODE", F.col("WND").getItem(2))\
    .withColumn("WND_SPEED", F.col("WND").getItem(3).cast(T.FloatType()))\
    .withColumn("WND_SPEED_QUALITY", F.col("WND").getItem(4).cast(T.IntegerType()))\
    .drop("WND")\
    
        
df_wind_split.show(20)

+--------+---------+---------+-----------+------------+-------+-------+-------+-----+---+----+------+-------+---------------+------------+---------+-----------------+
|LATITUDE|LONGITUDE|ELEVATION|        CIG|         VIS|    TMP|    DEW|    SLP|MONTH|DAY|HOUR|MINUTE|WND_DIR|WND_DIR_QUALITY|WND_DIR_CODE|WND_SPEED|WND_SPEED_QUALITY|
+--------+---------+---------+-----------+------------+-------+-------+-------+-----+---+----+------+-------+---------------+------------+---------+-----------------+
| 34.7728| -87.6399|    161.5|99999,9,9,N|999999,9,9,9|+0105,1|+9999,9|99999,9|    1|  1|   0|     0|  211.0|              1|           H|     18.0|                1|
| 34.7728| -87.6399|    161.5|99999,9,9,N|999999,9,9,9|+0100,1|+9999,9|99999,9|    1|  1|   0|     5|  214.0|              1|           H|     17.0|                1|
| 34.7728| -87.6399|    161.5|99999,9,9,N|999999,9,9,9|+0088,1|+9999,9|99999,9|    1|  1|   0|    10|  212.0|              1|           H|     17.0|                1

In [8]:
df_wind_filter = df_wind_split.filter((F.col("WND_DIR") <= 360) & (F.col("WND_DIR") >= 0))\
    .filter(F.col("WND_DIR_QUALITY").isin([0,1,4,5,9]))\
    .filter((F.col("WND_SPEED") >= 0) & (F.col("WND_SPEED") <= 900))\
    .filter(F.col("WND_SPEED_QUALITY").isin([0,1,4,5,9]))\
    .drop('WND_DIR_QUALITY', 'WND_SPEED_QUALITY')

df_wind_filter.show(20)

+--------+---------+---------+-----------+------------+-------+-------+-------+-----+---+----+------+-------+------------+---------+
|LATITUDE|LONGITUDE|ELEVATION|        CIG|         VIS|    TMP|    DEW|    SLP|MONTH|DAY|HOUR|MINUTE|WND_DIR|WND_DIR_CODE|WND_SPEED|
+--------+---------+---------+-----------+------------+-------+-------+-------+-----+---+----+------+-------+------------+---------+
| 34.7728| -87.6399|    161.5|99999,9,9,N|999999,9,9,9|+0105,1|+9999,9|99999,9|    1|  1|   0|     0|  211.0|           H|     18.0|
| 34.7728| -87.6399|    161.5|99999,9,9,N|999999,9,9,9|+0100,1|+9999,9|99999,9|    1|  1|   0|     5|  214.0|           H|     17.0|
| 34.7728| -87.6399|    161.5|99999,9,9,N|999999,9,9,9|+0088,1|+9999,9|99999,9|    1|  1|   0|    10|  212.0|           H|     17.0|
| 34.7728| -87.6399|    161.5|99999,9,9,N|999999,9,9,9|+0077,1|+9999,9|99999,9|    1|  1|   0|    15|  209.0|           H|     18.0|
| 34.7728| -87.6399|    161.5|99999,9,9,N|999999,9,9,9|+0076,1|+9999,

In [24]:
df_wind_filter.groupBy("WND_DIR_CODE").count().orderBy(F.desc("count")).show(10)

+------------+--------+
|WND_DIR_CODE|   count|
+------------+--------+
|           N|81887437|
|           V| 5931520|
|           9| 3860366|
|           H| 1159030|
+------------+--------+



In [ ]:
#Note that we should remove wind direction code which are missing ( unless there are calm winds)
df_wind_filter = df_wind_filter\
    .filter((F.col('WND_DIR_CODE') != '9') | (F.col('WND_SPEED') == 0))\
    .drop('WND_DIR_CODE')

# Note we can just directly drop WIND_DIR_CODE here without filtering, but just in case of
# of corrupted data.

In [10]:
df_wind_filter.printSchema()

root
 |-- LATITUDE: float (nullable = true)
 |-- LONGITUDE: float (nullable = true)
 |-- ELEVATION: float (nullable = true)
 |-- CIG: string (nullable = true)
 |-- VIS: string (nullable = true)
 |-- TMP: string (nullable = true)
 |-- DEW: string (nullable = true)
 |-- SLP: string (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- HOUR: integer (nullable = true)
 |-- MINUTE: integer (nullable = true)
 |-- WND_DIR: float (nullable = true)
 |-- WND_SPEED: float (nullable = true)



In [11]:
df_wind_filter.count()

89148961

### Ceiling Height

In [12]:
# Parse the CIG column.
# The only point of interested is the lowest ceiling height of obscuring phenomena layer /cloud cover. 
# The third field is metadata which is not important, and if it is missing it should not affect the analysis. 
# The last field signifies whether CAVOK (Ceiling And Visibility OK) has been reported by the system, which is a key indicator.

df_CIG_split = df_wind_filter.withColumn("CIG", F.split(F.col("CIG"), ","))\
    .withColumn("CIG_HEIGHT", F.col("CIG").getItem(0).cast(T.FloatType()))\
    .withColumn("CIG_HEIGHT_QUALITY",  F.col("CIG").getItem(1).cast(T.IntegerType()))\
    .withColumn("CIG_CAVOK", F.col("CIG").getItem(3))\
    .drop("CIG")


In [15]:
df_CIG_split.groupBy("CIG_CAVOK").count().show()

+---------+--------+
|CIG_CAVOK|   count|
+---------+--------+
|        N|74773918|
|        9| 9055693|
|        Y| 5319350|
+---------+--------+



Note that CIG_CAVOK is very imbalanced. Furthermore, according to our correlation analysis (see correlation analysis notebook) for CIG, we see that the CAVOK does not have significant correlation with the target value temperature. Hence, CIG CAVOK can be dropped. 

In [13]:
# filter the valid values for CIG distance and drop CAVOK
df_CIG_filter = df_CIG_split.filter(F.col("CIG_HEIGHT") != 99999.0)\
    .filter(F.col("CIG_HEIGHT_QUALITY").isin([0,1,4,5,9]))\
    .drop("CIG_CAVOK", "CIG_HEIGHT_QUALITY")

In [14]:
df_CIG_filter.printSchema()

root
 |-- LATITUDE: float (nullable = true)
 |-- LONGITUDE: float (nullable = true)
 |-- ELEVATION: float (nullable = true)
 |-- VIS: string (nullable = true)
 |-- TMP: string (nullable = true)
 |-- DEW: string (nullable = true)
 |-- SLP: string (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- HOUR: integer (nullable = true)
 |-- MINUTE: integer (nullable = true)
 |-- WND_DIR: float (nullable = true)
 |-- WND_SPEED: float (nullable = true)
 |-- CIG_HEIGHT: float (nullable = true)



In [18]:
### LEFT OFF HERE ####
df_CIG_filter.count()

48991387

### Visibility

In [15]:
# Parsing visibility VIS split

df_VIS = df_CIG_filter.withColumn("VIS", F.split(F.col("VIS"), ",")) \
    .withColumn("VIS_DISTANCE", F.col("VIS").getItem(0).cast(T.FloatType()))\
    .withColumn("VIS_DISTANCE_QUALITY", F.col("VIS").getItem(1).cast(T.IntegerType()))\
    .withColumn("VIS_VARIABILITY", F.col("VIS").getItem(2))\
    .withColumn("VIS_VARIABILITY_QUALITY", F.col("VIS").getItem(3).cast(T.IntegerType()))\
    .drop("VIS")

In [16]:
df_VIS_filter  = df_VIS.filter((F.col("VIS_DISTANCE") >= 0) & (F.col("VIS_DISTANCE") <= 160000))\
    .filter(F.col("VIS_DISTANCE_QUALITY").isin([0,1,4,5,9]))\
    .drop("VIS_DISTANCE_QUALITY") 

In [ ]:
df_VIS_filter.count()

48368223

In [ ]:
df_VIS_filter_2  = df_VIS_filter.filter(F.col("VIS_VARIABILITY").isin(['V', 'N'])) \
 .filter(F.col("VIS_VARIABILITY_QUALITY").isin([0,1,4,5,9]))\
.drop("VIS_VARIABILITY_QUALITY")

In [ ]:
df_VIS_filter_2.count()

22863341

Filtering valid valus of VIS_VARIABILITY significantly reduces the data size. Furthermore, based on correlation analysis performed on VIS and TMP, we conclude that we can drop VIS_VARIABILITY

In [17]:
df_VIS_filter = df_VIS_filter.drop("VIS_VARIABILITY", "VIS_VARIABILITY_QUALITY")

In [18]:
df_VIS_filter.printSchema()

root
 |-- LATITUDE: float (nullable = true)
 |-- LONGITUDE: float (nullable = true)
 |-- ELEVATION: float (nullable = true)
 |-- TMP: string (nullable = true)
 |-- DEW: string (nullable = true)
 |-- SLP: string (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- HOUR: integer (nullable = true)
 |-- MINUTE: integer (nullable = true)
 |-- WND_DIR: float (nullable = true)
 |-- WND_SPEED: float (nullable = true)
 |-- CIG_HEIGHT: float (nullable = true)
 |-- VIS_DISTANCE: float (nullable = true)



### Temperature

In [19]:
#Parsing temperature 
df_TEMP_split =  df_VIS_filter.withColumn('TMP' , F.split(F.col("TMP"), ",")) \
.withColumn('TMP_VALUE', F.col('TMP').getItem(0).cast(T.FloatType())) \
.withColumn('TMP_QUALITY', F.col('TMP').getItem(1)).drop('TMP')


In [ ]:
df_TEMP_split.groupBy('TMP_QUALITY').count().show()

+-----------+--------+
|TMP_QUALITY|   count|
+-----------+--------+
|          5|20163008|
|          6|   24769|
|          C| 2426907|
|          9|  184681|
|          A|    7690|
|          1|   37042|
|          7|   19131|
|          2|      84|
|          P|      29|
+-----------+--------+



In [20]:
df_TEMP_filter = df_TEMP_split.filter((F.col('TMP_VALUE') >= -932) & (F.col('TMP_VALUE') <= 618)).filter(F.col('TMP_QUALITY').isin(['0','1','4','5','9','A', 'C', 'I' ,'M', 'P', 'R', 'U']))\
    .drop('TMP_QUALITY')

In [21]:
df_TEMP_filter.printSchema()

root
 |-- LATITUDE: float (nullable = true)
 |-- LONGITUDE: float (nullable = true)
 |-- ELEVATION: float (nullable = true)
 |-- DEW: string (nullable = true)
 |-- SLP: string (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- HOUR: integer (nullable = true)
 |-- MINUTE: integer (nullable = true)
 |-- WND_DIR: float (nullable = true)
 |-- WND_SPEED: float (nullable = true)
 |-- CIG_HEIGHT: float (nullable = true)
 |-- VIS_DISTANCE: float (nullable = true)
 |-- TMP_VALUE: float (nullable = true)



In [22]:
df_TEMP_filter.count()

47967535

### Dew

In [23]:
# Parsing Dew
df_DEW_split = df_TEMP_filter.withColumn('DEW' , F.split(F.col("DEW"), ",")) \
.withColumn('DEW_VALUE', F.col('DEW').getItem(0).cast(T.FloatType())) \
.withColumn('DEW_QUALITY', F.col('DEW').getItem(1)).drop('DEW')

In [24]:
df_DEW_split.groupBy('DEW_QUALITY').count().show()

+-----------+--------+
|DEW_QUALITY|   count|
+-----------+--------+
|          1|25211851|
|          9|  221341|
|          5|20085725|
|          6|    6832|
|          C| 2417758|
|          2|   17299|
|          A|    6705|
|          P|      23|
|          7|       1|
+-----------+--------+



In [25]:
df_DEW_filter = df_DEW_split.filter((F.col('DEW_VALUE') >= -982) & (F.col('DEW_VALUE') <= 368)) \
.filter(F.col('DEW_QUALITY').isin(['0','1','4','5','9','A', 'C', 'I' ,'M', 'P', 'R', 'U'])) \
.drop('DEW_QUALITY')

In [26]:
df_DEW_filter.printSchema()

root
 |-- LATITUDE: float (nullable = true)
 |-- LONGITUDE: float (nullable = true)
 |-- ELEVATION: float (nullable = true)
 |-- SLP: string (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- HOUR: integer (nullable = true)
 |-- MINUTE: integer (nullable = true)
 |-- WND_DIR: float (nullable = true)
 |-- WND_SPEED: float (nullable = true)
 |-- CIG_HEIGHT: float (nullable = true)
 |-- VIS_DISTANCE: float (nullable = true)
 |-- TMP_VALUE: float (nullable = true)
 |-- DEW_VALUE: float (nullable = true)



In [27]:
df_DEW_filter.count()

47722062

### SLP

In [28]:
# Parsing SLP
df_SLP_split = df_DEW_filter.withColumn('SLP' , F.split(F.col("SLP"), ",")) \
.withColumn('SLP_VALUE', F.col('SLP').getItem(0).cast(T.FloatType())) \
.withColumn('SLP_QUALITY', F.col('SLP').getItem(1).cast(T.IntegerType())).drop('SLP')

In [29]:
df_SLP_split.groupBy('SLP_QUALITY').count().show()

+-----------+--------+
|SLP_QUALITY|   count|
+-----------+--------+
|          1|10385725|
|          9|29592369|
|          2|    7578|
|          6|    2612|
|          5| 7733778|
+-----------+--------+



In [30]:
df_SLP_filter = df_SLP_split.filter(F.col('SLP_QUALITY').isin([0,1,4,5,9]))\
    .filter((F.col('SLP_VALUE') >= 8600) & (F.col('SLP_VALUE') <= 10900))\
    .drop('SLP_QUALITY')

In [31]:
df_SLP_filter.count()

18119503

### Save Clean Data

In [ ]:
write_directory = "2024_cleaned_data/"
df_SLP_filter.write.parquet(
    path=write_directory, 
    mode="overwrite", 
    compression="snappy" 
)


25/10/29 15:29:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/10/29 15:29:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/10/29 15:29:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
25/10/29 15:29:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
25/10/29 15:29:59 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
25/10/29 15:29:59 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/10/29 15:30:00 WARN MemoryManager: Total allocation exceeds 95.00%